<a href="https://colab.research.google.com/github/Chaitanya-2004-code/TeleBot/blob/main/TeleBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys

# Remove every cached telegram-related module from this running process
for mod in list(sys.modules):
    if mod == "telegram" or mod.startswith("telegram."):
        del sys.modules[mod]

!rm -rf /usr/local/lib/python3.12/dist-packages/telegram*
!pip install --no-cache-dir --force-reinstall "python-telegram-bot==22.8"

import importlib
import telegram
importlib.reload(telegram)
print(telegram.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 769.4/769.4 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 273.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 206.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.9/124.9 kB 245.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 233.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.3/133.3 kB 288.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 181.7 MB/s eta 0:00:00
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.0
  Attempting uninstall: idna
    Found existing installation: idna 3.18
    Uninstalling idna-3.18:
      Successfully uninstalled idna-3.18
  Attempting uninstall: h11
    Found existing installation: h11 0.16.0
    Uninstalling h11-0.16

22.8


In [1]:
!pip install --no-cache-dir --force-reinstall "APScheduler==3.10.4"

import importlib
import apscheduler
importlib.reload(apscheduler)
print(apscheduler.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 kB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.1/510.1 kB 50.7 MB/s eta 0:00:00
  Attempting uninstall: pytz
    Found existing installation: pytz 2026.2
    Uninstalling pytz-2026.2:
      Successfully uninstalled pytz-2026.2
  Attempting uninstall: tzlocal
    Found existing installation: tzlocal 5.4.3
    Uninstalling tzlocal-5.4.3:
      Successfully uninstalled tzlocal-5.4.3
  Attempting uninstall: six
    Found existing installation: six 1.17.0
    Uninstalling six-1.17.0:
      Successfully uninstalled six-1.17.0
  Attempting uninstall: APScheduler
    Found existing installation: APScheduler 3.10.4
    Uninstalling APScheduler-3.10.4:
      Successfully uninstalled APScheduler-3.10.4


3.10.4


In [1]:
%%writefile recon_bot.py
import re
import asyncio
from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, ContextTypes, filters

BOT_TOKEN = "8952765447:AAH_xV5I8yGDdnw5i9Tc7Tov-fsm8sbRQmg"

HOSTNAME_RE = re.compile(r"^(?!-)[A-Za-z0-9-]{1,63}(?<!-)(\.(?!-)[A-Za-z0-9-]{1,63}(?<!-))*$")


async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text("Welcome {}".format(update.effective_user.first_name))


async def message_handler(update: Update, context: ContextTypes.DEFAULT_TYPE):
    target = update.message.text.strip()

    if not HOSTNAME_RE.match(target):
        await update.message.reply_text("Invalid hostname/IP.")
        return

    try:
        # Replaces subprocess.run non-blockingly with a 10 second timeout
        proc = await asyncio.wait_for(
            asyncio.create_subprocess_exec(
                "host", "-t", "a", target,
                stdout=asyncio.subprocess.PIPE,
                stderr=asyncio.subprocess.PIPE
            ),
            timeout=10.0
        )

        # Non-blocking wait for stdout data
        stdout, _ = await proc.communicate()
        out = stdout.decode().strip()

        await update.message.reply_text(out[:3500] or "(no output)")

    except asyncio.TimeoutError:
        await update.message.reply_text("Error: Command timed out after 10 seconds.")
    except Exception as e:
        await update.message.reply_text(f"An unexpected error occurred: {str(e)}")


if __name__ == '__main__':
    app = Application.builder().token(BOT_TOKEN).build()
    app.add_handler(CommandHandler("start", start))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, message_handler))
    app.run_polling()

Writing recon_bot.py


In [2]:
!nohup python3 recon_bot.py > bot.log 2>&1 &

